In [2]:
import subprocess
from pathlib import Path

input_tif = Path("meanpm25_india_wustl_2023.tif")
output_dir = Path("compressed")
output_dir.mkdir(exist_ok=True)

compressions = {
    "DEFLATE": [
        "-co", "COMPRESS=DEFLATE",
        "-co", "PREDICTOR=3",
        "-co", "ZLEVEL=9",
        "-co", "TILED=YES",
    ],

    "ZSTD": [
        "-co", "COMPRESS=ZSTD",
        "-co", "PREDICTOR=3",
        "-co", "ZSTD_LEVEL=22",
        "-co", "TILED=YES",
    ],

    "LZW": [
        "-co", "COMPRESS=LZW",
        "-co", "PREDICTOR=3",
        "-co", "TILED=YES",
    ],

    "PACKBITS": [
        "-co", "COMPRESS=PACKBITS",
        "-co", "TILED=YES",
    ],
}

results = []

for name, options in compressions.items():

    output = output_dir / f"{input_tif.stem}_{name.lower()}.tif"

    cmd = [
        "gdal_translate",
        str(input_tif),
        str(output),
        "-of", "GTiff",
        *options,
    ]

    print(f"Running {name}...")
    subprocess.run(cmd, check=True)

    size_mb = output.stat().st_size / (1024 ** 2)

    results.append((name, size_mb, output))

print("\nResults:")
print("-" * 50)

for name, size, path in sorted(results, key=lambda x: x[1]):
    print(f"{name:12} {size:10.2f} MB  {path}")

print("\nSmallest:")
best = min(results, key=lambda x: x[1])
print(f"{best[0]} → {best[1]:.2f} MB")

Running DEFLATE...
Input file size is 2923, 3034
0...10...20...30...40...50...60...70...80...90...100 - done.
Running ZSTD...
Input file size is 2923, 3034
0...10...20...30...40...50...60...70...80...90...100 - done.
Running LZW...
Input file size is 2923, 3034
0...10...20...30...40...50...60...70...80...90...100 - done.
Running PACKBITS...
Input file size is 2923, 3034
0...10...20...30...40...50...60...70...80...90...100 - done.

Results:
--------------------------------------------------
ZSTD               7.07 MB  compressed/meanpm25_india_wustl_2023_zstd.tif
DEFLATE            7.47 MB  compressed/meanpm25_india_wustl_2023_deflate.tif
LZW                9.71 MB  compressed/meanpm25_india_wustl_2023_lzw.tif
PACKBITS          34.13 MB  compressed/meanpm25_india_wustl_2023_packbits.tif

Smallest:
ZSTD → 7.07 MB
